# Add OS / OS.time to Merged Datasets

Run this **once** before running the MB script.

Joins `OS` and `OS.time` from `outcome.csv` into all 8 merged files.

**Note:** OS is overall survival (not 5-year binary) — AUC-5yr is derived from it using a threshold.

In [2]:
"""
Add OS and OS.time to all 8 merged datasets.
Run this ONCE — then MB script will find them directly in the files.
"""
import pandas as pd
from pathlib import Path

try:
    SCRIPT_DIR = Path(__file__).resolve().parent
except NameError:
    cwd = Path.cwd()
    SCRIPT_DIR = cwd if cwd.name == "MERGE" else cwd / "MERGE"

MERGE_DIR = SCRIPT_DIR if SCRIPT_DIR.name == "MERGE" else SCRIPT_DIR.parent / "MERGE"

# Find outcome.csv
OUTCOME_CANDIDATES = [
    SCRIPT_DIR.parent.parent / "data" / "outcome.csv",
    SCRIPT_DIR.parent / "RNA" / "preprocessed" / "outcome.csv",
    SCRIPT_DIR.parent.parent / "RNA" / "preprocessed" / "outcome.csv",
    MERGE_DIR / "outcome.csv",
]

outcome = None
for cand in OUTCOME_CANDIDATES:
    if cand.exists():
        outcome = pd.read_csv(cand, index_col=0)
        print(f"Loaded outcome from: {cand}")
        break

if outcome is None:
    print("ERROR: outcome.csv not found. Searched:")
    for c in OUTCOME_CANDIDATES: print(f"  {c}")
else:
    # Normalize outcome index to patient ID (TCGA-XX-XXXX)
    outcome.index = [
        "-".join(str(i).replace(".", "-").split("-")[:3])
        for i in outcome.index
    ]
    print(f"Outcome shape: {outcome.shape}")
    print(f"Columns: {outcome.columns.tolist()}")
    print(f"Sample index: {outcome.index[:3].tolist()}")
    print()

    DATASET_NAMES = [
        "01_ultra_conservative", "02_conservative", "03_standard",
        "04_fdr_significant",    "05_balanced",     "06_correlation",
        "07_top_correlated",     "08_composite",
    ]

    for name in DATASET_NAMES:
        ds_file = MERGE_DIR / f"{name}.csv"
        if not ds_file.exists():
            print(f"Not found: {name}")
            continue

        df = pd.read_csv(ds_file, index_col=0)

        if "OS" in df.columns and "OS.time" in df.columns:
            print(f"Already has OS: {name}  ({df.shape})")
            continue

        # Join
        df_joined = df.join(outcome[["OS", "OS.time"]], how="inner")
        n_lost = len(df) - len(df_joined)
        df_joined.to_csv(ds_file)
        print(f"Updated {name}: {df.shape} -> {df_joined.shape}  (lost {n_lost} samples without outcome)")

    print("\nDone. All datasets now have OS and OS.time.")

Loaded outcome from: C:\Users\olegk\Desktop\Thesis_v3\data\outcome.csv
Outcome shape: (1076, 2)
Columns: ['OS.time', 'OS']
Sample index: ['TCGA-C8-A275', 'TCGA-AC-A7VC', 'TCGA-BH-A1F8']

Updated 01_ultra_conservative: (526, 444) -> (526, 446)  (lost 0 samples without outcome)
Updated 02_conservative: (526, 444) -> (526, 446)  (lost 0 samples without outcome)
Updated 03_standard: (526, 444) -> (526, 446)  (lost 0 samples without outcome)
Updated 04_fdr_significant: (526, 444) -> (526, 446)  (lost 0 samples without outcome)
Updated 05_balanced: (526, 444) -> (526, 446)  (lost 0 samples without outcome)
Updated 06_correlation: (526, 444) -> (526, 446)  (lost 0 samples without outcome)
Updated 07_top_correlated: (526, 444) -> (526, 446)  (lost 0 samples without outcome)
Updated 08_composite: (526, 444) -> (526, 446)  (lost 0 samples without outcome)

Done. All datasets now have OS and OS.time.
